Congrats! You just graduated UVA's BSDS program and got a job working at a movie studio in Hollywood. 

Your boss is the head of the studio and wants to know if they can gain a competitive advantage by predicting new movies that might get high imdb scores (movie rating). 

You would like to be able to explain the model to mere mortals but need a fairly robust and flexible approach so you've chosen to use decision trees to get started. 

In doing so, similar to  great data scientists of the past you remembered the excellent education provided to you at UVA in a undergrad data science course and have outline 20ish steps that will need to be undertaken to complete this task. As always, you will need to make sure to #comment your work heavily. 

 Footnotes: 
-	You can add or combine steps if needed
-	Also, remember to try several methods during evaluation and always be mindful of how the model will be used in practice.
- Make sure all your variables are the correct type (factor, character,numeric, etc.)

In [1]:
import pandas as pd
import numpy as np

In [4]:
#1. Load the data
#Sometimes need to set the working directory back out of a folder that we create a file in

#import os
#os.listdir()
#print(os.getcwd())
#os.chdir('c:\\Users\\Brian Wright\\Documents\\3001Python\\DS-3001')

movie_metadata=pd.read_csv("/workspaces/DS-3021/data/movie_metadata.csv")
movie_metadata


,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5038,Color,Scott Smith,1.0,87.0,2.0,318.0,Daphne Zuniga,637.0,NaN,Comedy|Drama,...,6.0,English,Canada,NaN,NaN,2013.0,470.0,7.7,NaN,84
5039,Color,NaN,43.0,43.0,NaN,319.0,Valorie Curry,841.0,NaN,Crime|Drama|Mystery|Thriller,...,359.0,English,USA,TV-14,NaN,NaN,593.0,7.5,16.00,32000
5040,Color,Benjamin Roberds,13.0,76.0,0.0,0.0,Maxwell Moody,0.0,NaN,Drama|Horror|Thriller,...,3.0,English,USA,NaN,1400.0,2013.0,0.0,6.3,NaN,16
5041,Color,Daniel Hsia,14.0,100.0,0.0,489.0,Daniel Henney,946.0,10443.0,Comedy|Drama|Romance,...,9.0,English,USA,PG-13,NaN,2012.0,719.0,6.3,2.35,660


In [20]:
movies = movie_metadata[['movie_title', 'duration', 'genres', 'gross', 'country', 'content_rating', 'budget', 'cast_total_facebook_likes', 'imdb_score']]
movies.head(10)

,movie_title,duration,genres,gross,country,content_rating,budget,cast_total_facebook_likes,imdb_score
0,Avatar,178.0,Action|Adventure|Fantasy|Sci-Fi,760505847.0,USA,PG-13,237000000.0,4834,7.9
1,Pirates of the Caribbean: At World's End,169.0,Action|Adventure|Fantasy,309404152.0,USA,PG-13,300000000.0,48350,7.1
2,Spectre,148.0,Action|Adventure|Thriller,200074175.0,UK,PG-13,245000000.0,11700,6.8
3,The Dark Knight Rises,164.0,Action|Thriller,448130642.0,USA,PG-13,250000000.0,106759,8.5
4,Star Wars: Episode VII - The Force Awakens ...,NaN,Documentary,NaN,NaN,NaN,NaN,143,7.1
5,John Carter,132.0,Action|Adventure|Sci-Fi,73058679.0,USA,PG-13,263700000.0,1873,6.6
6,Spider-Man 3,156.0,Action|Adventure|Romance,336530303.0,USA,PG-13,258000000.0,46055,6.2
7,Tangled,100.0,Adventure|Animation|Comedy|Family|Fantasy|Musi...,200807262.0,USA,PG,260000000.0,2036,7.8
8,Avengers: Age of Ultron,141.0,Action|Adventure|Sci-Fi,458991599.0,USA,PG-13,250000000.0,92000,7.5
9,Harry Potter and the Half-Blood Prince,153.0,Adventure|Family|Fantasy|Mystery,301956980.0,UK,PG,250000000.0,58753,7.5


In [21]:
pd.options.mode.chained_assignment = None  # default='warn'

# Initialize genre columns with zeros
genres = ['Action', 'Drama', 'Comedy', 'Romance', 'Horror', 'Thriller', 'Adventure', 'Fantasy', 'Sci-Fi', 'Animation', 'Documentary', 'Family', 'Mystery', 'Western', 'Crime', 'Music', 'War', 'History', 'Sport', 'Short', 'Biography']
for genre in genres:
    movies[genre] = 0

# Update genre columns based on the presence of each genre in the 'genres' string
for index, row in movies.iterrows():
    for genre in genres:
        if genre in row['genres']:
            movies.loc[index, genre] = 1


In [22]:
movies = movies.drop(columns=['genres'])
movies = movies.dropna()
movies = movies.reset_index(drop=True)
movies

,movie_title,duration,gross,country,content_rating,budget,cast_total_facebook_likes,imdb_score,Action,Drama,...,Family,Mystery,Western,Crime,Music,War,History,Sport,Short,Biography
0,Avatar,178.0,760505847.0,USA,PG-13,237000000.0,4834,7.9,1,0,...,0,0,0,0,0,0,0,0,0,0
1,Pirates of the Caribbean: At World's End,169.0,309404152.0,USA,PG-13,300000000.0,48350,7.1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,Spectre,148.0,200074175.0,UK,PG-13,245000000.0,11700,6.8,1,0,...,0,0,0,0,0,0,0,0,0,0
3,The Dark Knight Rises,164.0,448130642.0,USA,PG-13,250000000.0,106759,8.5,1,0,...,0,0,0,0,0,0,0,0,0,0
4,John Carter,132.0,73058679.0,USA,PG-13,263700000.0,1873,6.6,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3835,Primer,77.0,424760.0,USA,PG-13,7000.0,368,7.0,0,1,...,0,0,0,0,0,0,0,0,0,0
3836,Cavite,80.0,70071.0,Philippines,Not Rated,7000.0,0,6.3,0,0,...,0,0,0,0,0,0,0,0,0,0
3837,El Mariachi,81.0,2040920.0,USA,R,7000.0,147,6.9,1,1,...,0,0,0,1,0,0,0,0,0,0
3838,Newlyweds,95.0,4584.0,USA,Not Rated,9000.0,690,6.4,0,1,...,0,0,0,0,0,0,0,0,0,0


In [23]:
movies = movies[movies['budget'] < 3500000000]
movies = movies[movies['cast_total_facebook_likes'] < 320000]
movies 

,movie_title,duration,gross,country,content_rating,budget,cast_total_facebook_likes,imdb_score,Action,Drama,...,Family,Mystery,Western,Crime,Music,War,History,Sport,Short,Biography
0,Avatar,178.0,760505847.0,USA,PG-13,237000000.0,4834,7.9,1,0,...,0,0,0,0,0,0,0,0,0,0
1,Pirates of the Caribbean: At World's End,169.0,309404152.0,USA,PG-13,300000000.0,48350,7.1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,Spectre,148.0,200074175.0,UK,PG-13,245000000.0,11700,6.8,1,0,...,0,0,0,0,0,0,0,0,0,0
3,The Dark Knight Rises,164.0,448130642.0,USA,PG-13,250000000.0,106759,8.5,1,0,...,0,0,0,0,0,0,0,0,0,0
4,John Carter,132.0,73058679.0,USA,PG-13,263700000.0,1873,6.6,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3835,Primer,77.0,424760.0,USA,PG-13,7000.0,368,7.0,0,1,...,0,0,0,0,0,0,0,0,0,0
3836,Cavite,80.0,70071.0,Philippines,Not Rated,7000.0,0,6.3,0,0,...,0,0,0,0,0,0,0,0,0,0
3837,El Mariachi,81.0,2040920.0,USA,R,7000.0,147,6.9,1,1,...,0,0,0,1,0,0,0,0,0,0
3838,Newlyweds,95.0,4584.0,USA,Not Rated,9000.0,690,6.4,0,1,...,0,0,0,0,0,0,0,0,0,0


In [24]:
for index, row in movies.iterrows():
    if row['imdb_score'] >= 7.5:
        movies.loc[index, 'imdb_score'] = 'excellent'
    else:
        movies.loc[index, 'imdb_score'] = 'ok'    
movies

/tmp/ipykernel_9087/422855009.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'excellent' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  movies.loc[index, 'imdb_score'] = 'excellent'


,movie_title,duration,gross,country,content_rating,budget,cast_total_facebook_likes,imdb_score,Action,Drama,...,Family,Mystery,Western,Crime,Music,War,History,Sport,Short,Biography
0,Avatar,178.0,760505847.0,USA,PG-13,237000000.0,4834,excellent,1,0,...,0,0,0,0,0,0,0,0,0,0
1,Pirates of the Caribbean: At World's End,169.0,309404152.0,USA,PG-13,300000000.0,48350,ok,1,0,...,0,0,0,0,0,0,0,0,0,0
2,Spectre,148.0,200074175.0,UK,PG-13,245000000.0,11700,ok,1,0,...,0,0,0,0,0,0,0,0,0,0
3,The Dark Knight Rises,164.0,448130642.0,USA,PG-13,250000000.0,106759,excellent,1,0,...,0,0,0,0,0,0,0,0,0,0
4,John Carter,132.0,73058679.0,USA,PG-13,263700000.0,1873,ok,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3835,Primer,77.0,424760.0,USA,PG-13,7000.0,368,ok,0,1,...,0,0,0,0,0,0,0,0,0,0
3836,Cavite,80.0,70071.0,Philippines,Not Rated,7000.0,0,ok,0,0,...,0,0,0,0,0,0,0,0,0,0
3837,El Mariachi,81.0,2040920.0,USA,R,7000.0,147,ok,1,1,...,0,0,0,1,0,0,0,0,0,0
3838,Newlyweds,95.0,4584.0,USA,Not Rated,9000.0,690,ok,0,1,...,0,0,0,0,0,0,0,0,0,0


In [25]:
movies.dtypes

movie_title                   object
duration                     float64
gross                        float64
country                       object
content_rating                object
budget                       float64
cast_total_facebook_likes      int64
imdb_score                    object
Action                         int64
Drama                          int64
Comedy                         int64
Romance                        int64
Horror                         int64
Thriller                       int64
Adventure                      int64
Fantasy                        int64
Sci-Fi                         int64
Animation                      int64
Documentary                    int64
Family                         int64
Mystery                        int64
Western                        int64
Crime                          int64
Music                          int64
War                            int64
History                        int64
Sport                          int64
S

2 Ensure all the variables are classified correctly including the target variable and collapse factor variables as needed.

3 Check for missing variables and correct as needed. Once you've completed the cleaning again create a function that will do this for you in the future. In the submission, include only the function and the function call.

4 Guess what, you don't need to scale the data, because DTs don't require this to be done, they make local greedy decisions...keeps getting easier, go to the next step.

5 Determine the baserate or prevalence for the classifier, what does this number mean?

6 Split your data into test, tune, and train. (80/10/10)

7 Create the kfold object for cross validation.

8 Create the scoring metric you will use to evaluate your model and the max depth hyperparameter (grid search) 

9 Build the classifier object 

10 Use the kfold object and the scoring metric to find the best hyperparameter value for max depth via the grid search method.

11 Fit the model to the training data.

12 What is the best depth value?

13 Print out the model

14 View the results, comment on how the model performed using the metrics you selected.

15 Which variables appear to be contributing the most (variable importance) 

16 Use the predict method on the tune data and print out the results.

17 How does the model perform on the tune data?

18 Print out the confusion matrix for the tune data, what does it tell you about the model?

19 What are the top 3 movies based on the tune set? Which variables are most important in predicting the top 3 movies?

20 Use a different hyperparameter for the grid search function and go through the process above again using the tune set. 

21 Did the model improve with the new hyperparameter search?

22 Using the better model, predict the test data and print out the results.

23 Summarize what you learned along the way and make recommendations to your boss on how this could be used moving forward, being careful not to over promise.